# Orientation and orphan diagnostic

This notebook audits the retained legacy MultiProt/FiberDock pipeline and tests whether its two implicit orientation branches are logically justified. It is an executable diagnostic log, not a production-pipeline change.

## Questions

1. Does the legacy implementation expose an orientation option?
2. What is the difference between a raw orphan, a threshold-qualified orphan, an orientation mismatch, and a contact/clash failure?
3. Do both implicit orientations produce candidates on real retained data?
4. Do the imposed thresholds and clash rules explain the observed losses?

All conclusions are labeled as observations, inferences, or hypotheses. The notebook reads existing repository/job artifacts and writes no derived data outside this notebook.

## tl;dr

The legacy CLI has no user-facing `orientation` argument, but `transformationFiltering.py` constructs two implicit assignments: `o1 = (left→template-chain-1, right→template-chain-2)` and `o2 = (right→template-chain-1, left→template-chain-2)`. This notebook measures those branches independently.

The key accounting rule is: a pair is an **orphan** only when one required side has a qualified alignment and its required partner does not. A pair whose two raw records exist but both fail the match/hotspot/coverage gate is **not** an orphan. Contact filtering and clash filtering are later stages and are reported separately.

## Context and reproducibility

The primary evidence is the retained job under `working_version/Multiprot-new/prism-fiberdock-cli/jobs/10pairs-kutem`. The current source tree and the existing `transformation_gate_ablation.ipynb` are inspected for provenance, but the real-data ledger uses the retained legacy artifacts because the current scratch run directory is empty.

The notebook intentionally reimplements only the small, observable predicates needed for diagnosis. It does not silently claim that this is a full rerun of the legacy pipeline: geometry replay is a selected-case, Cα-only audit of the legacy transform/clash logic.

In [1]:
from __future__ import annotations

import configparser
import hashlib
import itertools
import math
import pickle
import re
import sys
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

import pandas as pd
from IPython.display import display

SOURCE_REPO = Path("/scratch/rshadi25/GitHub/PRISM-prescript")
LEGACY_ROOT = SOURCE_REPO / "working_version/Multiprot-new/prism-fiberdock-cli"
LEGACY_JOB = LEGACY_ROOT / "jobs/10pairs-kutem"
EXISTING_GATE_NOTEBOOK = SOURCE_REPO / "notebooks/transformation_gate_ablation.ipynb"
CURRENT_SOURCE = SOURCE_REPO / "src/transformation.py"
CURRENT_CLI = SOURCE_REPO / "prism.py"
LEGACY_FILTER = LEGACY_ROOT / "run_files/transformationFiltering.py"
LEGACY_INI = LEGACY_ROOT / "prism.ini"
LEGACY_ALIGNMENT = LEGACY_JOB / "alignment"
LEGACY_PREPROCESS = LEGACY_JOB / "preprocess"
LEGACY_TEMPLATE = LEGACY_ROOT / "template"

SELECTED_PAIR = ("1gfi", "2bv1")
SELECTED_TEMPLATES = ("1zx4AB", "1ngmEF")
RUN_GEOMETRY_REPLAY = True

assert "gtalign_env" in sys.executable, f"Run this notebook with gtalign_env; got {sys.executable}"
print("Interpreter:", sys.executable)
print("Python/project roots:")
print("  SOURCE_REPO:", SOURCE_REPO)
print("  LEGACY_JOB:", LEGACY_JOB)
print("  Existing gate notebook:", EXISTING_GATE_NOTEBOOK.exists())
print("  Current source:", CURRENT_SOURCE.exists())
print("  Legacy alignment directory:", LEGACY_ALIGNMENT.exists())


Interpreter: /home/rshadi25/.conda/envs/gtalign_env/bin/python
Python/project roots:
  SOURCE_REPO: /scratch/rshadi25/GitHub/PRISM-prescript
  LEGACY_JOB: /scratch/rshadi25/GitHub/PRISM-prescript/working_version/Multiprot-new/prism-fiberdock-cli/jobs/10pairs-kutem
  Existing gate notebook: True
  Current source: True
  Legacy alignment directory: True


In [2]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()[:16]

provenance_paths = [
    EXISTING_GATE_NOTEBOOK, CURRENT_SOURCE, CURRENT_CLI, LEGACY_FILTER,
    LEGACY_INI, LEGACY_JOB / "lists/pair_list", LEGACY_JOB / "lists/template_list",
]
provenance = []
for path in provenance_paths:
    provenance.append({
        "path": str(path.relative_to(SOURCE_REPO)) if path.is_relative_to(SOURCE_REPO) else str(path),
        "exists": path.exists(),
        "bytes": path.stat().st_size if path.exists() else None,
        "sha256_16": sha256(path) if path.exists() else None,
    })
display(pd.DataFrame(provenance))

cli_text = CURRENT_CLI.read_text(errors="replace") if CURRENT_CLI.exists() else ""
legacy_cli_text = (LEGACY_ROOT / "prism.py").read_text(errors="replace")
filter_text = LEGACY_FILTER.read_text(errors="replace")

implementation_facts = pd.DataFrame([
    {"implementation": "legacy CLI user-facing orientation argument", "present": bool(re.search(r"orientation|orient", legacy_cli_text, re.I)), "evidence": "legacy prism.py positional CLI"},
    {"implementation": "legacy filter implicit o1/o2 branches", "present": "left1" in filter_text and "left2" in filter_text, "evidence": "transformationFiltering.py"},
    {"implementation": "current CLI user-facing orientation argument", "present": bool(re.search(r"orientation|orient", cli_text, re.I)), "evidence": "current prism.py"},
    {"implementation": "current source internal branch handling", "present": bool(re.search(r"orientation|left2|right2", CURRENT_SOURCE.read_text(errors="replace"), re.I)) if CURRENT_SOURCE.exists() else False, "evidence": "src/transformation.py"},
])
display(implementation_facts)


,path,exists,bytes,sha256_16
0,notebooks/transformation_gate_ablation.ipynb,True,74654,ff9157124e2cd5e0
1,src/transformation.py,True,20138,520dcad61a69f05e
2,prism.py,True,16665,bf54977823753707
3,working_version/Multiprot-new/prism-fiberdock-...,True,17252,e32404b5d95146dd
4,working_version/Multiprot-new/prism-fiberdock-...,True,2223,85ff860241d6448c
5,working_version/Multiprot-new/prism-fiberdock-...,True,109,6ae95d3bd635250f
6,working_version/Multiprot-new/prism-fiberdock-...,True,147504,237d2e3bec8e0370


,implementation,present,evidence
0,legacy CLI user-facing orientation argument,False,legacy prism.py positional CLI
1,legacy filter implicit o1/o2 branches,True,transformationFiltering.py
2,current CLI user-facing orientation argument,False,current prism.py
3,current source internal branch handling,True,src/transformation.py


## 1. Load the retained job and parse the legacy thresholds

The values below are read from `prism.ini`; they are not chosen by this notebook. The later sensitivity analysis varies them only as a diagnostic and does not modify the pipeline configuration.

In [3]:
config = configparser.ConfigParser()
config.read(LEGACY_INI)
section = config["Transformation_Filtering"]

def cfg_int(name):
    return int(section[name])

def cfg_float(name):
    return float(section[name])

THRESHOLDS = {
    "multiprotcount": cfg_int("multiprotcount"),
    "min_match_count": cfg_int("minimum_residue_match_count"),
    "min_match_percentage": cfg_float("minimum_residue_match_percentage"),
    "minimum_hotspot": cfg_int("minimum_hotspot_match_number"),
    "diff_percentage": cfg_float("diff_percentage"),
    "template_residue_boundary": cfg_int("template_residue_count"),
    "contact_count": cfg_int("contact_count"),
    "clash_distance": cfg_int("clashing_distance"),
    "max_clash_count": cfg_int("max_clashing_count"),
    "hotspot_criterion": cfg_int("hotspotcriterion"),
    "hotspot_count": cfg_int("hotspotcount"),
}

pair_list_path = LEGACY_JOB / "lists/pair_list"
template_list_path = LEGACY_JOB / "lists/template_list"
PAIR_LIST = [tuple(line.split()[:2]) for line in pair_list_path.read_text().splitlines() if line.split()]
TEMPLATE_LIST = [line.strip() for line in template_list_path.read_text().splitlines() if line.strip()]

print("Legacy thresholds:")
display(pd.DataFrame([THRESHOLDS]))
print(f"Pair-list rows: {len(PAIR_LIST):,}; template-list rows: {len(TEMPLATE_LIST):,}; unique templates in list: {len(set(TEMPLATE_LIST)):,}")
display(pd.DataFrame(PAIR_LIST, columns=["left_query", "right_query"]))


Legacy thresholds:


,multiprotcount,min_match_count,min_match_percentage,minimum_hotspot,diff_percentage,template_residue_boundary,contact_count,clash_distance,max_clash_count,hotspot_criterion,hotspot_count
0,3,15,50.0,1,20.0,50,5,3,5,2,1


Pair-list rows: 10; template-list rows: 21,072; unique templates in list: 21,072


,left_query,right_query
0,1cew,2ghuD
1,2uwjG,2uwjE
2,7unrH,8cd1X
3,1fqj,1fqjI
4,1bpb,3k77
5,1gfi,2bv1
6,1c3d,2gom
7,1acb,1acbE
8,1bvn,1bvnA
9,1dfj,1dfjE


In [4]:
ALIGNMENT_RE = re.compile(r"^(?P<template>[A-Za-z0-9]{6})_(?P<chain>[A-Za-z0-9])_(?P<query>.+)$")

def load_alignment(path: Path):
    with path.open("rb") as handle:
        payload = pickle.load(handle, encoding="latin1")
    # Retained legacy jobs contain both list and dict containers. Dict keys are
    # solution indices; sorting them reproduces the list order used downstream.
    if isinstance(payload, dict):
        return [payload[key] for key in sorted(payload)]
    return payload

alignment_inventory = []
malformed_alignment_files = []
for path in sorted(LEGACY_ALIGNMENT.iterdir()):
    if not path.is_file() or path.suffix == ".multiprot":
        continue
    match = ALIGNMENT_RE.match(path.name)
    if not match:
        malformed_alignment_files.append(path.name)
        continue
    try:
        payload = load_alignment(path)
        n_solutions = len(payload) if isinstance(payload, (list, tuple)) else None
        alignment_inventory.append({
            "path": path,
            "filename": path.name,
            "template": match.group("template"),
            "template_chain": match.group("chain"),
            "query": match.group("query"),
            "n_solutions": n_solutions,
        })
    except Exception as exc:
        malformed_alignment_files.append(f"{path.name}: {type(exc).__name__}: {exc}")

alignment_inventory = pd.DataFrame(alignment_inventory)
print(f"Parseable alignment records: {len(alignment_inventory):,}")
print(f"Observed query IDs: {alignment_inventory['query'].nunique():,}; observed template IDs: {alignment_inventory['template'].nunique():,}")
print(f"Malformed/unreadable files: {len(malformed_alignment_files):,}")
if malformed_alignment_files:
    print(malformed_alignment_files[:5])

ALIGNMENT_PATHS = {(r.template, r.template_chain, r.query): r.path for r in alignment_inventory.itertuples()}
ALIGNMENT_PAYLOADS = {}
for key, path in ALIGNMENT_PATHS.items():
    ALIGNMENT_PAYLOADS[key] = load_alignment(path)

record_counts = alignment_inventory.groupby(["template", "template_chain"]).size().reset_index(name="records")
display(record_counts.head())


Parseable alignment records: 4,463
Observed query IDs: 3; observed template IDs: 837
Malformed/unreadable files: 0


,template,template_chain,records
0,1a0hDE,D,3
1,1a0hDE,E,3
2,1a79BC,B,2
3,1a79BC,C,2
4,1aorAB,A,2


## 2. Define orientation, threshold, and contact predicates

For a template identifier such as `1zx4AB`, the first and second template chains are `A` and `B`. The legacy code builds two possible assignments:

- `o1`: left query → chain A, right query → chain B
- `o2`: right query → chain A, left query → chain B

The functions below keep raw record presence separate from solution qualification. This is essential for diagnosing whether a missing candidate is caused by orientation, thresholding, contact geometry, or clash filtering.

In [5]:
def template_chains(template: str):
    if len(template) != 6:
        raise ValueError(f"Expected six-character template ID, got {template!r}")
    return template[4], template[5]

def orientation_requirements(left_query: str, right_query: str, template: str, orientation: str):
    chain1, chain2 = template_chains(template)
    if orientation == "o1":
        return [(left_query, chain1), (right_query, chain2)]
    if orientation == "o2":
        return [(right_query, chain1), (left_query, chain2)]
    raise ValueError(orientation)

def alignment_key(query: str, template: str, chain: str):
    return template, chain, query

def raw_payload(query: str, template: str, chain: str):
    return ALIGNMENT_PAYLOADS.get(alignment_key(query, template, chain))

def interface_size(match_dict):
    return len(match_dict)

def target_residue_number(target_residue: str):
    match = re.search(r"(-?\d+)", str(target_residue))
    return int(match.group(1)) if match else None

@lru_cache(maxsize=None)
def hotspot_ids(template: str):
    path = LEGACY_TEMPLATE / "hotspot" / f"hotspot{template}"
    if not path.exists():
        return tuple()
    return tuple(line.strip().split()[0] for line in path.read_text(errors="replace").splitlines() if line.strip() and not line.lstrip().startswith("#"))

@lru_cache(maxsize=None)
def template_interface_size(template: str, chain: str):
    path = LEGACY_TEMPLATE / "interfaces" / f"{template}_{chain}.int"
    if not path.exists():
        return 0
    return sum(1 for line in path.read_text(errors="replace").splitlines() if line.strip())

@lru_cache(maxsize=None)
def template_contact_ids(template: str):
    path = LEGACY_TEMPLATE / "contact" / f"{template}.txt"
    if not path.exists():
        return tuple()
    keys = []
    for line in path.read_text(errors="replace").splitlines():
        fields = line.split()
        if len(fields) >= 2:
            # Legacy interfaceMatchCheck looks up left_key + right_key.
            keys.append(fields[0] + fields[1])
    return tuple(keys)

def same_residue_type(left: str, right: str):
    # Legacy hotspot criterion 2 compares the residue-type character at index 2.
    return len(str(left)) > 2 and len(str(right)) > 2 and str(left)[2] == str(right)[2]

def solution_metrics(query: str, template: str, chain: str, solution):
    match_count, ref_mol, transform_vector, match_dict = solution
    size = interface_size(match_dict)
    template_size = template_interface_size(template, chain)
    # The legacy implementation uses the number of lines in the chain-specific
    # interface file as proteinSize. Keep an observed-residue fallback only for
    # incomplete retained artifacts.
    if not template_size:
        template_size = len({str(k).split(".")[-1] for k in match_dict})
    coverage = 100.0 * match_count / template_size if template_size else 0.0
    hotspot_matches = 0
    hotspots = set(hotspot_ids(template))
    if hotspots:
        for template_residue, target_residue in match_dict.items():
            if str(template_residue) in hotspots and same_residue_type(template_residue, target_residue):
                hotspot_matches += 1
    match_pass = match_count >= THRESHOLDS["min_match_count"]
    hotspot_pass = hotspot_matches >= THRESHOLDS["hotspot_count"]
    required_coverage = (THRESHOLDS["min_match_percentage"] - THRESHOLDS["diff_percentage"]) if template_size > THRESHOLDS["template_residue_boundary"] else THRESHOLDS["min_match_percentage"]
    coverage_pass = coverage > required_coverage
    return {
        "query": query, "template": template, "template_chain": chain,
        "match_count": match_count, "interface_size": size,
        "template_size": template_size, "coverage_pct": coverage,
        "required_coverage_pct": required_coverage,
        "hotspot_matches": hotspot_matches,
        "match_pass": match_pass, "hotspot_pass": hotspot_pass,
        "coverage_pass": coverage_pass,
        "threshold_pass": match_pass and hotspot_pass and coverage_pass,
        "payload": solution,
    }

def qualified_solution_ids(query: str, template: str, chain: str, *, min_count=None, min_pct=None, diff_pct=None):
    payload = raw_payload(query, template, chain) or []
    ids = []
    for idx, solution in enumerate(payload):
        metrics = solution_metrics(query, template, chain, solution)
        count_limit = THRESHOLDS["min_match_count"] if min_count is None else min_count
        if metrics["template_size"] > THRESHOLDS["template_residue_boundary"]:
            pct_limit = (THRESHOLDS["min_match_percentage"] if min_pct is None else min_pct) - (THRESHOLDS["diff_percentage"] if diff_pct is None else diff_pct)
        else:
            pct_limit = THRESHOLDS["min_match_percentage"] if min_pct is None else min_pct
        count_ok = metrics["match_count"] >= count_limit
        pct_ok = metrics["coverage_pct"] > pct_limit
        hotspot_ok = metrics["hotspot_matches"] >= THRESHOLDS["hotspot_count"]
        if count_ok and pct_ok and hotspot_ok:
            ids.append(idx)
    return ids

def complementary_contact_count(first_metrics, second_metrics):
    first_match = first_metrics["payload"][3]
    second_match = second_metrics["payload"][3]
    contact_keys = set(template_contact_ids(first_metrics["template"]))
    return sum(1 for left in first_match for right in second_match if left + right in contact_keys)


In [6]:
def evaluate_orientation(left_query: str, right_query: str, template: str, orientation: str):
    requirements = orientation_requirements(left_query, right_query, template, orientation)
    first_query, first_chain = requirements[0]
    second_query, second_chain = requirements[1]
    first_raw = raw_payload(first_query, template, first_chain)
    second_raw = raw_payload(second_query, template, second_chain)
    first_ids = qualified_solution_ids(first_query, template, first_chain) if first_raw is not None else []
    second_ids = qualified_solution_ids(second_query, template, second_chain) if second_raw is not None else []

    contact_pairs = []
    for first_id, second_id in itertools.product(first_ids, second_ids):
        first_metrics = solution_metrics(first_query, template, first_chain, first_raw[first_id])
        second_metrics = solution_metrics(second_query, template, second_chain, second_raw[second_id])
        contacts = complementary_contact_count(first_metrics, second_metrics)
        contact_pairs.append({
            "first_solution": first_id, "second_solution": second_id,
            "contact_count": contacts,
            "contact_pass": contacts >= THRESHOLDS["contact_count"],
        })

    # Search for evidence that the missing qualified side is qualified on the
    # same template chain under the opposite query assignment. That pattern is
    # an orientation mismatch, not proof that the pair is biologically valid.
    alternate_evidence = []
    for missing_query, missing_chain in requirements:
        if qualified_solution_ids(missing_query, template, missing_chain):
            continue
        for chain in template_chains(template):
            if chain == missing_chain:
                continue
            ids = qualified_solution_ids(missing_query, template, chain)
            if ids:
                alternate_evidence.append({"query": missing_query, "chain": chain, "qualified_ids": ids})

    both_raw = first_raw is not None and second_raw is not None
    n_qualified = int(bool(first_ids)) + int(bool(second_ids))
    if both_raw and first_ids and second_ids:
        status = "contact_pairable" if any(p["contact_pass"] for p in contact_pairs) else "contact_failed"
        reason = "both_required_sides_qualified"
    elif both_raw and not first_ids and not second_ids:
        status, reason = "both_sides_threshold_failed", "raw_records_present_but_no_qualified_solution"
    elif n_qualified == 1:
        status = "qualified_orphan"
        reason = "orientation_mismatch" if alternate_evidence else "missing_or_failed_partner"
    elif first_raw is not None or second_raw is not None:
        status, reason = "raw_orphan", "one_required_raw_record_missing"
    else:
        status, reason = "not_observed", "no_required_raw_records"

    return {
        "left_query": left_query, "right_query": right_query, "template": template,
        "orientation": orientation, "first_query": first_query, "first_chain": first_chain,
        "second_query": second_query, "second_chain": second_chain,
        "first_raw": first_raw is not None, "second_raw": second_raw is not None,
        "first_raw_n": len(first_raw) if first_raw is not None else 0,
        "second_raw_n": len(second_raw) if second_raw is not None else 0,
        "first_qualified_ids": tuple(first_ids), "second_qualified_ids": tuple(second_ids),
        "n_contact_combinations": len(contact_pairs),
        "n_contact_pass": sum(p["contact_pass"] for p in contact_pairs),
        "alternate_evidence": tuple((x["query"], x["chain"], tuple(x["qualified_ids"])) for x in alternate_evidence),
        "status": status, "reason": reason,
    }


## 3. Real-data orientation ledger

The ledger is restricted to query pairs from the retained pair list and template IDs observed in the retained alignment directory. This avoids treating an absent file as evidence about the entire database.

In [7]:
observed_templates = sorted(alignment_inventory["template"].unique())
ledger_rows = []
for left_query, right_query in PAIR_LIST:
    for template in observed_templates:
        for orientation in ("o1", "o2"):
            row = evaluate_orientation(left_query, right_query, template, orientation)
            if row["first_raw"] or row["second_raw"]:
                ledger_rows.append(row)

ledger = pd.DataFrame(ledger_rows)
print(f"Orientation ledger rows with >=1 required raw record: {len(ledger):,}")
print("Status counts:")
display(ledger["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="rows"))
print("Reason counts:")
display(ledger["reason"].value_counts(dropna=False).rename_axis("reason").reset_index(name="rows"))

selected_ledger = ledger[
    ledger["left_query"].eq(SELECTED_PAIR[0])
    & ledger["right_query"].eq(SELECTED_PAIR[1])
    & ledger["template"].isin(SELECTED_TEMPLATES)
].copy()
selected_columns = [
    "left_query", "right_query", "template", "orientation", "first_query", "first_chain",
    "second_query", "second_chain", "first_raw_n", "second_raw_n",
    "first_qualified_ids", "second_qualified_ids", "n_contact_combinations",
    "n_contact_pass", "status", "reason", "alternate_evidence",
]
display(selected_ledger[selected_columns].sort_values(["template", "orientation"]))


Orientation ledger rows with >=1 required raw record: 3,348
Status counts:


,status,rows
0,raw_orphan,1812
1,both_sides_threshold_failed,948
2,qualified_orphan,565
3,contact_pairable,19
4,contact_failed,4


Reason counts:


,reason,rows
0,one_required_raw_record_missing,1812
1,raw_records_present_but_no_qualified_solution,948
2,missing_or_failed_partner,526
3,orientation_mismatch,39
4,both_required_sides_qualified,23


,left_query,right_query,template,orientation,first_query,first_chain,second_query,second_chain,first_raw_n,second_raw_n,first_qualified_ids,second_qualified_ids,n_contact_combinations,n_contact_pass,status,reason,alternate_evidence
1886,1gfi,2bv1,1ngmEF,o1,1gfi,E,2bv1,F,3,3,"(0,)",(),0,0,qualified_orphan,orientation_mismatch,"((2bv1, E, (0, 1)),)"
1887,1gfi,2bv1,1ngmEF,o2,2bv1,E,1gfi,F,3,3,"(0, 1)","(0,)",2,2,contact_pairable,both_required_sides_qualified,()
2162,1gfi,2bv1,1zx4AB,o1,1gfi,A,2bv1,B,3,3,"(0,)","(0, 1, 2)",3,3,contact_pairable,both_required_sides_qualified,()
2163,1gfi,2bv1,1zx4AB,o2,2bv1,A,1gfi,B,3,3,"(0, 1)","(1, 2)",4,4,contact_pairable,both_required_sides_qualified,()


### Interpretation boundary

A raw alignment record means that a MultiProt result file exists and can be loaded. A qualified solution additionally passes the legacy match-count, hotspot, and coverage tests. A contact-qualified combination additionally passes the complementary-contact count. A clash-qualified result would require the subsequent transform/overlap stage. These are different events and are intentionally counted separately.

In [8]:
ledger_summary = pd.DataFrame({
    "metric": [
        "raw one-sided rows", "raw both-sided rows",
        "qualified orphan rows", "qualified orientation-mismatch rows",
        "both-sided threshold-failed rows", "contact-pairable rows", "contact-failed rows",
    ],
    "value": [
        int((ledger["first_raw"] ^ ledger["second_raw"]).sum()),
        int((ledger["first_raw"] & ledger["second_raw"]).sum()),
        int((ledger["status"] == "qualified_orphan").sum()),
        int(((ledger["status"] == "qualified_orphan") & ledger["reason"].eq("orientation_mismatch")).sum()),
        int((ledger["status"] == "both_sides_threshold_failed").sum()),
        int((ledger["status"] == "contact_pairable").sum()),
        int((ledger["status"] == "contact_failed").sum()),
    ],
})
display(ledger_summary)


,metric,value
0,raw one-sided rows,2233
1,raw both-sided rows,1115
2,qualified orphan rows,565
3,qualified orientation-mismatch rows,39
4,both-sided threshold-failed rows,948
5,contact-pairable rows,19
6,contact-failed rows,4


## 4. Mechanical tests for the classification logic

These tests use small in-memory payloads to verify the accounting semantics. They do not claim biological validity. In particular, a same-chain hit is evidence for a possible orientation mismatch, not a proof that swapping chains is correct.

In [9]:
def synthetic_solution(match_count=20, template_size=40, residue_prefix="A"):
    match_dict = {f"T.L.{100+i}": f"{residue_prefix}.L.{200+i}" for i in range(template_size)}
    return [match_count, "A", [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], match_dict]

def classify_presence(orientation_presence):
    # orientation_presence is (first_raw, second_raw, first_qualified, second_qualified).
    first_raw, second_raw, first_q, second_q = orientation_presence
    if first_q and second_q:
        return "qualified_pair"
    if first_q ^ second_q:
        return "qualified_orphan"
    if first_raw ^ second_raw:
        return "raw_orphan"
    if first_raw and second_raw:
        return "both_sides_threshold_failed"
    return "not_observed"

assert classify_presence((True, True, True, True)) == "qualified_pair"
assert classify_presence((True, True, True, False)) == "qualified_orphan"
assert classify_presence((True, False, False, False)) == "raw_orphan"
assert classify_presence((True, True, False, False)) == "both_sides_threshold_failed"
assert classify_presence((False, False, False, False)) == "not_observed"

synthetic_logic_tests = pd.DataFrame([
    {"case": "both orientation assignments have evidence", "o1": (True, True, True, True), "o2": (True, True, True, True), "expected": "both pairable before contacts"},
    {"case": "only o1 qualified", "o1": (True, True, True, True), "o2": (True, True, False, False), "expected": "o2 is threshold-failed, not orphan"},
    {"case": "one raw record only", "o1": (True, False, False, False), "o2": (False, True, False, False), "expected": "raw orphan in each branch"},
    {"case": "same-chain alternative", "o1": (True, True, True, False), "o2": (True, True, False, True), "expected": "orientation mismatch is a diagnostic reason"},
])
display(synthetic_logic_tests)
print("Synthetic classification assertions passed.")


,case,o1,o2,expected
0,both orientation assignments have evidence,"(True, True, True, True)","(True, True, True, True)",both pairable before contacts
1,only o1 qualified,"(True, True, True, True)","(True, True, False, False)","o2 is threshold-failed, not orphan"
2,one raw record only,"(True, False, False, False)","(False, True, False, False)",raw orphan in each branch
3,same-chain alternative,"(True, True, True, False)","(True, True, False, True)",orientation mismatch is a diagnostic reason


Synthetic classification assertions passed.


## 5. Threshold sensitivity on selected real cases

This is a local sensitivity probe. It asks whether the selected observations are robust to plausible threshold changes. It does not recommend changing thresholds without a benchmark/ground-truth analysis.

In [10]:
sensitivity_rows = []
for template in SELECTED_TEMPLATES:
    for orientation in ("o1", "o2"):
        requirements = orientation_requirements(*SELECTED_PAIR, template, orientation)
        q1, c1 = requirements[0]
        q2, c2 = requirements[1]
        for min_count, min_pct, diff_pct in [(8, 30.0, 20), (15, 50.0, 20), (15, 50.0, 35), (20, 50.0, 20)]:
            ids1 = qualified_solution_ids(q1, template, c1, min_count=min_count, min_pct=min_pct, diff_pct=diff_pct)
            ids2 = qualified_solution_ids(q2, template, c2, min_count=min_count, min_pct=min_pct, diff_pct=diff_pct)
            contact_pass = 0
            raw1, raw2 = raw_payload(q1, template, c1), raw_payload(q2, template, c2)
            if raw1 is not None and raw2 is not None:
                for i, j in itertools.product(ids1, ids2):
                    m1 = solution_metrics(q1, template, c1, raw1[i])
                    m2 = solution_metrics(q2, template, c2, raw2[j])
                    contact_pass += complementary_contact_count(m1, m2) >= THRESHOLDS["contact_count"]
            sensitivity_rows.append({
                "template": template, "orientation": orientation,
                "min_match_count": min_count, "minimum_match_pct": min_pct, "diff_pct": diff_pct,
                "first_qualified": len(ids1), "second_qualified": len(ids2),
                "qualified_combinations": len(ids1) * len(ids2), "contact_pass": contact_pass,
            })
display(pd.DataFrame(sensitivity_rows))


,template,orientation,min_match_count,minimum_match_pct,diff_pct,first_qualified,second_qualified,qualified_combinations,contact_pass
0,1zx4AB,o1,8,30.0,20,1,3,3,3
1,1zx4AB,o1,15,50.0,20,1,3,3,3
2,1zx4AB,o1,15,50.0,35,1,3,3,3
3,1zx4AB,o1,20,50.0,20,1,1,1,1
4,1zx4AB,o2,8,30.0,20,2,2,4,4
5,1zx4AB,o2,15,50.0,20,2,2,4,4
6,1zx4AB,o2,15,50.0,35,2,2,4,4
7,1zx4AB,o2,20,50.0,20,1,0,0,0
8,1ngmEF,o1,8,30.0,20,2,1,2,2
9,1ngmEF,o1,15,50.0,20,1,0,0,0


## 6. Selected legacy transform/clash replay

The legacy `overlap()` routine transforms the two target structures and counts cross-partner Cα distances below 3 Å; it accepts when the count is below 5. The replay below ports that narrow calculation for the selected contact-qualified combinations.

This is intentionally labeled **Cα-only**. It is useful for localizing whether the legacy clash gate is the immediate bottleneck, but it is not an all-atom steric validation and cannot establish that a candidate is physically plausible.

In [11]:
def read_ca_coordinates(path: Path):
    coords = []
    with path.open(errors="replace") as handle:
        for line in handle:
            if line.startswith("ATOM  ") and line[12:16].strip() == "CA":
                try:
                    coords.append((float(line[30:38]), float(line[38:46]), float(line[46:54])))
                except ValueError:
                    pass
    return coords

def legacy_rotation(phi, theta, psi):
    """Port of rotationDictExtractor(), with rows preserved as in legacy code."""
    return [
        [math.cos(theta) * math.cos(psi), math.cos(theta) * math.sin(psi), -math.sin(theta)],
        [-math.cos(phi) * math.sin(psi) + math.sin(phi) * math.sin(theta) * math.cos(psi), math.cos(phi) * math.cos(psi) + math.sin(phi) * math.sin(theta) * math.sin(psi), math.sin(phi) * math.cos(theta)],
        [math.sin(phi) * math.sin(psi) + math.cos(phi) * math.sin(theta) * math.cos(psi), -math.sin(phi) * math.cos(psi) + math.cos(phi) * math.sin(theta) * math.sin(psi), math.cos(phi) * math.cos(theta)],
    ]

def legacy_transpose_rotation(phi, theta, psi):
    """Port of transposeRotationDictExtractor(), with rows preserved."""
    return [
        [math.cos(theta) * math.cos(psi), -math.cos(phi) * math.sin(psi) + math.sin(phi) * math.sin(theta) * math.cos(psi), math.sin(phi) * math.sin(psi) + math.cos(phi) * math.sin(theta) * math.cos(psi)],
        [math.cos(theta) * math.sin(psi), math.cos(phi) * math.cos(psi) + math.sin(phi) * math.sin(theta) * math.sin(psi), -math.sin(phi) * math.cos(psi) + math.cos(phi) * math.sin(theta) * math.sin(psi)],
        [-math.sin(theta), math.sin(phi) * math.cos(theta), math.cos(phi) * math.cos(theta)],
    ]

def apply_legacy_transform(coords, transform_vector, ref_mol):
    # Legacy pdbTransform() adds pi to all Euler angles. For refMol == 0 it
    # translates after rotation; for refMol == 1 it subtracts translation before
    # the transpose rotation. The multiplication below intentionally uses
    # matrix columns, matching the legacy scalar expressions.
    phi, theta, psi, tx, ty, tz = transform_vector
    phi, theta, psi = (value + math.pi for value in (phi, theta, psi))
    if ref_mol == 0:
        matrix = legacy_rotation(phi, theta, psi)
        add_translation = True
    elif ref_mol == 1:
        matrix = legacy_transpose_rotation(phi, theta, psi)
        add_translation = False
    else:
        raise ValueError(f"Unexpected legacy refMol={ref_mol!r}")
    transformed = []
    for x, y, z in coords:
        if add_translation:
            transformed.append((
                x * matrix[0][0] + y * matrix[1][0] + z * matrix[2][0] + tx,
                x * matrix[0][1] + y * matrix[1][1] + z * matrix[2][1] + ty,
                x * matrix[0][2] + y * matrix[1][2] + z * matrix[2][2] + tz,
            ))
        else:
            x, y, z = x - tx, y - ty, z - tz
            transformed.append((
                x * matrix[0][0] + y * matrix[1][0] + z * matrix[2][0],
                x * matrix[0][1] + y * matrix[1][1] + z * matrix[2][1],
                x * matrix[0][2] + y * matrix[1][2] + z * matrix[2][2],
            ))
    return transformed

def clash_summary(first_coords, second_coords, distance=3.0, max_count=5):
    d2 = distance * distance
    count = 0
    min_d2 = float("inf")
    for x1, y1, z1 in first_coords:
        for x2, y2, z2 in second_coords:
            value = (x1-x2)**2 + (y1-y2)**2 + (z1-z2)**2
            min_d2 = min(min_d2, value)
            if value < d2:
                count += 1
    return {
        "clash_count": count,
        "min_distance_A": math.sqrt(min_d2) if min_d2 < float("inf") else None,
        "clash_pass": count < max_count,
    }

def selected_contact_combinations(template, orientation):
    row = evaluate_orientation(*SELECTED_PAIR, template, orientation)
    req = orientation_requirements(*SELECTED_PAIR, template, orientation)
    first_q, first_chain = req[0]
    second_q, second_chain = req[1]
    raw1, raw2 = raw_payload(first_q, template, first_chain), raw_payload(second_q, template, second_chain)
    records = []
    for i, j in itertools.product(row["first_qualified_ids"], row["second_qualified_ids"]):
        m1 = solution_metrics(first_q, template, first_chain, raw1[i])
        m2 = solution_metrics(second_q, template, second_chain, raw2[j])
        contacts = complementary_contact_count(m1, m2)
        if contacts >= THRESHOLDS["contact_count"]:
            records.append((req, m1, m2, contacts))
    return records

if RUN_GEOMETRY_REPLAY:
    geometry_rows = []
    for template in SELECTED_TEMPLATES:
        for orientation in ("o1", "o2"):
            for req, m1, m2, contacts in selected_contact_combinations(template, orientation):
                first_coords = read_ca_coordinates(LEGACY_PREPROCESS / f"{m1['query']}.pdb")
                second_coords = read_ca_coordinates(LEGACY_PREPROCESS / f"{m2['query']}.pdb")
                first_payload, second_payload = m1["payload"], m2["payload"]
                transformed_first = apply_legacy_transform(first_coords, first_payload[2], first_payload[1])
                transformed_second = apply_legacy_transform(second_coords, second_payload[2], second_payload[1])
                clash = clash_summary(transformed_first, transformed_second, THRESHOLDS["clash_distance"], THRESHOLDS["max_clash_count"])
                geometry_rows.append({
                    "template": template, "orientation": orientation,
                    "first_query": m1["query"], "second_query": m2["query"],
                    "first_solution": first_payload, "second_solution": second_payload,
                    "contact_count": contacts, **clash,
                })
    geometry = pd.DataFrame(geometry_rows)
    if not geometry.empty:
        display(geometry[["template", "orientation", "first_query", "second_query", "contact_count", "clash_count", "min_distance_A", "clash_pass"]])
        display(geometry.groupby(["template", "orientation"], as_index=False).agg(
            candidates=("clash_count", "size"),
            min_clash_count=("clash_count", "min"),
            max_clash_count=("clash_count", "max"),
            min_distance_A=("min_distance_A", "min"),
            clash_passes=("clash_pass", "sum"),
        ))
    else:
        print("No selected contact-qualified combinations were available for geometry replay.")


,template,orientation,first_query,second_query,contact_count,clash_count,min_distance_A,clash_pass
0,1zx4AB,o1,1gfi,2bv1,18,49,1.200184,False
1,1zx4AB,o1,1gfi,2bv1,9,62,0.552026,False
2,1zx4AB,o1,1gfi,2bv1,9,61,0.671552,False
3,1zx4AB,o2,2bv1,1gfi,13,67,0.753748,False
4,1zx4AB,o2,2bv1,1gfi,8,66,0.300548,False
5,1zx4AB,o2,2bv1,1gfi,8,108,0.767435,False
6,1zx4AB,o2,2bv1,1gfi,7,79,0.801312,False
7,1ngmEF,o2,2bv1,1gfi,12,70,0.818522,False
8,1ngmEF,o2,2bv1,1gfi,15,71,0.827210,False


,template,orientation,candidates,min_clash_count,max_clash_count,min_distance_A,clash_passes
0,1ngmEF,o2,2,70,71,0.818522,0
1,1zx4AB,o1,3,49,62,0.552026,0
2,1zx4AB,o2,4,66,108,0.300548,0


## 7. Sanity checks and findings

These checks are deliberately mechanical. They verify that the ledger does not duplicate its identity keys, that only the two intended branches are present, and that selected paths exist. They do not validate the biology.

In [12]:
assert set(ledger["orientation"].unique()).issubset({"o1", "o2"})
assert not ledger.duplicated(["left_query", "right_query", "template", "orientation"]).any()
assert all((LEGACY_ALIGNMENT / name).exists() for name in [])
assert LEGACY_ALIGNMENT.exists() and LEGACY_PREPROCESS.exists()
assert all((LEGACY_PREPROCESS / f"{query}.pdb").exists() for query in SELECTED_PAIR)

# Confirm that the selected real case is represented in the ledger.
assert not selected_ledger.empty
print("Ledger, path, and selected-case assertions passed.")

findings = pd.DataFrame([
    {"label": "OBSERVATION", "finding": "The legacy CLI has no explicit orientation option; the filter creates two implicit assignments."},
    {"label": "OBSERVATION", "finding": f"The retained job contains {len(alignment_inventory):,} parseable alignment records across {alignment_inventory['template'].nunique():,} observed templates."},
    {"label": "INFERENCE", "finding": "A qualified orphan is a branch-specific missing partner after thresholds; raw presence alone is insufficient to call an orphan."},
    {"label": "OBSERVATION", "finding": "The selected real case is retained in the ledger and exposes orientation-specific qualification/contact outcomes."},
    {"label": "LIMITATION", "finding": "The replay is Cα-only and selected-case; it does not replace all-atom clash scoring or a ground-truth benchmark."},
])
display(findings)


Ledger, path, and selected-case assertions passed.


,label,finding
0,OBSERVATION,The legacy CLI has no explicit orientation opt...
1,OBSERVATION,"The retained job contains 4,463 parseable alig..."
2,INFERENCE,A qualified orphan is a branch-specific missin...
3,OBSERVATION,The selected real case is retained in the ledg...
4,LIMITATION,The replay is Cα-only and selected-case; it do...


## Takeaways

- **Orientation is a candidate-generation branch, not an optional scoring parameter.** The legacy implementation tries both query-to-template-chain assignments because the pair-list order does not by itself establish which query belongs to which template chain.
- **Orientation can reduce candidates if one branch is selected too early.** On the retained data, branch-specific alignment availability and qualification are not symmetric; therefore a one-orientation implementation can discard candidates before contact or clash scoring.
- **“Orphan” needs a stage-qualified definition.** Missing raw files, raw records that fail thresholds, qualified one-sided assignments, contact failures, and clash failures are distinct bottlenecks.
- **Threshold logic should be benchmarked before changing defaults.** The sensitivity table is a diagnostic, not evidence that a relaxed threshold is scientifically acceptable.
- **The next decisive experiment is a full per-stage ledger over the complete benchmark:** raw records → qualified solutions → contact pairs → transformed candidates → clash outcomes → downstream scoring, with query/template/chain/orientation preserved in every row.

## Suggested next step

Use the ledger schema from this notebook as the monitoring contract for a full pipeline run. Run both `o1` and `o2` through the same downstream stages, keep candidates that fail each stage in an explicit status table, and compare recall against known positives before imposing any orientation or threshold default.

## 8. Parameterized current-pipeline comparison

This section turns the diagnostic questions into reproducible current-pipeline arms. Edit the configuration cell to select a CSV, a template subset, and threshold overrides. The `native_default` arm omits `--orientation` and therefore tests the current default; `o1` and `o2` are explicit fixed-assignment arms.

Each opt-in run is placed under a new isolated directory with its own `processed/`, stage-status JSONL, candidate audit, stdout, and stderr. `RUN_CONFIGURED_PIPELINES` is `False` by default.

In [ ]:
import json
import os
import shlex
import subprocess
import time

sys.path.insert(0, str(SOURCE_REPO))
from src.pipeline_inputs import select_templates
from src.transformation_config import TransformationThresholds

# User-editable run configuration. Keep the source input/template assets read-only.
PIPELINE_INPUT_CSV = SOURCE_REPO / "inputs.csv"
PIPELINE_TEMPLATE_IDS = list(SELECTED_TEMPLATES)
PIPELINE_TEMPLATE_LIST = None  # Optional Path to a plain-text six-character template manifest.
PIPELINE_ARMS = {"native_default": None, "o1": "o1", "o2": "o2"}
PIPELINE_ALIGNER = "tmalign"
PIPELINE_THRESHOLD_OVERRIDES = {}
PIPELINE_OUTPUT_ROOT = SOURCE_REPO / "tmp/agent/notebook-orientation-comparison"
RUN_CONFIGURED_PIPELINES = False

if PIPELINE_TEMPLATE_LIST is not None:
    selected_pipeline_templates = select_templates(
        [], template_list_path=PIPELINE_TEMPLATE_LIST
    )
else:
    selected_pipeline_templates = select_templates(
        [], explicit_templates=PIPELINE_TEMPLATE_IDS
    )
configured_thresholds = TransformationThresholds.from_environment().with_overrides(
    PIPELINE_THRESHOLD_OVERRIDES
)

configuration_rows = [{
    "input_csv": str(PIPELINE_INPUT_CSV),
    "templates": ",".join(selected_pipeline_templates),
    "arms": ",".join(PIPELINE_ARMS),
    "aligner": PIPELINE_ALIGNER,
    "threshold_overrides": json.dumps(PIPELINE_THRESHOLD_OVERRIDES, sort_keys=True),
    "execution_enabled": RUN_CONFIGURED_PIPELINES,
}]
display(pd.DataFrame(configuration_rows))
print("Input exists:", PIPELINE_INPUT_CSV.exists())
print("Selected templates:", selected_pipeline_templates)

In [ ]:
def pipeline_command(arm_name: str, run_root: Path):
    """Build one current PRISM command with explicit provenance inputs."""
    orientation = PIPELINE_ARMS[arm_name]
    command = [
        sys.executable, str(CURRENT_CLI),
        "--inputs-csv", str(PIPELINE_INPUT_CSV),
        "--templates", *selected_pipeline_templates,
        "--aligner", PIPELINE_ALIGNER,
        "--no-refine",
        "--candidate-audit-path", str(run_root / "status/candidates.jsonl"),
    ]
    if orientation is not None:
        command.extend(["--orientation", orientation])
    for field, value in PIPELINE_THRESHOLD_OVERRIDES.items():
        command.extend(["--" + field.replace("_", "-"), str(value)])
    return command

command_rows = []
_preview_root = PIPELINE_OUTPUT_ROOT / "preview"
for arm_name in PIPELINE_ARMS:
    command_rows.append({
        "arm": arm_name,
        "orientation_option": PIPELINE_ARMS[arm_name] or "(omitted: native default)",
        "command": shlex.join(pipeline_command(arm_name, _preview_root / arm_name)),
    })
display(pd.DataFrame(command_rows))

In [ ]:
def _prepare_isolated_pipeline_root(run_root: Path):
    run_root.mkdir(parents=True, exist_ok=True)
    for name in ("templates", "external_tools"):
        link = run_root / name
        if not link.exists() and not link.is_symlink():
            link.symlink_to(SOURCE_REPO / name, target_is_directory=True)

def _transformation_pair_count(run_root: Path):
    transformation = run_root / "processed/transformation"
    left = {path.name[:-6] for path in transformation.glob("*_L.pdb")}
    right = {path.name[:-6] for path in transformation.glob("*_R.pdb")}
    return len(left & right)

def run_parameterized_comparison():
    """Run all configured arms in fresh roots and return terminal summaries."""
    if not PIPELINE_INPUT_CSV.exists():
        raise FileNotFoundError(PIPELINE_INPUT_CSV)
    run_root = PIPELINE_OUTPUT_ROOT / time.strftime("%Y%m%d-%H%M%S")
    results = []
    for arm_name in PIPELINE_ARMS:
        arm_root = run_root / arm_name
        _prepare_isolated_pipeline_root(arm_root)
        status_path = arm_root / "status/stages.jsonl"
        status_path.parent.mkdir(parents=True, exist_ok=True)
        command = pipeline_command(arm_name, arm_root)
        environment = os.environ.copy()
        environment["PRISM_STAGE_STATUS_PATH"] = str(status_path)
        environment["PRISM_RUN_ID"] = arm_name + "-" + time.strftime("%Y%m%d%H%M%S")
        started = time.time()
        completed = subprocess.run(
            command, cwd=arm_root, env=environment,
            text=True, capture_output=True, check=False,
        )
        (arm_root / "status/stdout.log").write_text(completed.stdout, encoding="utf-8")
        (arm_root / "status/stderr.log").write_text(completed.stderr, encoding="utf-8")
        events = []
        if status_path.exists():
            events = [json.loads(line) for line in status_path.read_text().splitlines() if line.strip()]
        results.append({
            "arm": arm_name,
            "orientation": PIPELINE_ARMS[arm_name] or "native",
            "return_code": completed.returncode,
            "elapsed_seconds": round(time.time() - started, 2),
            "completed_stages": ",".join(
                event["stage"] for event in events if event.get("event") == "completed"
            ),
            "transformation_pairs": _transformation_pair_count(arm_root),
            "run_root": str(arm_root),
        })
    return pd.DataFrame(results)

if RUN_CONFIGURED_PIPELINES:
    comparison_results = run_parameterized_comparison()
    display(comparison_results)
else:
    comparison_results = pd.DataFrame()
    print("Execution disabled. Set RUN_CONFIGURED_PIPELINES = True after reviewing the command table.")

### Threshold sensitivity plan

The same runner can be reused for a small pre-registered threshold matrix. Keep the orientation, CSV, template panel, aligner, and output-root policy fixed; change one threshold family at a time and retain failed-stage records. A changed candidate count is an observation, not evidence that the new threshold is biologically better.

In [ ]:
THRESHOLD_SWEEP = {
    "current_defaults": {},
    "lower_match_count": {"minimum_residue_match_count": 12},
    "lower_tm_score": {"tm_score_threshold": 0.4},
    "relaxed_clash_count": {"max_clashing_count": 8},
}

sweep_rows = []
for sweep_name, overrides in THRESHOLD_SWEEP.items():
    sweep_thresholds = TransformationThresholds.from_environment().with_overrides(overrides)
    sweep_rows.append({
        "sweep": sweep_name,
        "minimum_residue_match_count": sweep_thresholds.minimum_residue_match_count,
        "tm_score_threshold": sweep_thresholds.tm_score_threshold,
        "max_clashing_count": sweep_thresholds.max_clashing_count,
        "interpretation": "candidate-yield probe only; validate against benchmark labels",
    })
display(pd.DataFrame(sweep_rows))